In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install torchmetrics[image] torch-fidelity

In [ ]:
# ==============================================================================  
# Phase I: Environment Setup and Dataset Engineering  
# ==============================================================================  
import os 
import random 
import numpy as np 
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader 
from torchvision import transforms 
from PIL import Image 
import matplotlib.pyplot as plt 
import timm 
import torchmetrics 

# Ensure deterministic execution for reproducibility  
def seed_everything(seed=42):     
    random.seed(seed)    
    os.environ['PYTHONHASHSEED'] = str(seed)    
    np.random.seed(seed)    
    torch.manual_seed(seed)    
    torch.cuda.manual_seed(seed)    
    torch.backends.cudnn.deterministic = True     
    torch.backends.cudnn.benchmark = False 

seed_everything(42) 

# Configure T4 GPU Environment 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = torch.cuda.amp.GradScaler()  # Automatic Mixed Precision for T4 optimization  

# Define Kaggle Dataset Paths 
BASE_DIR = "/kaggle/input/datasets/koushikkumardinda/multi-class-pneumonia-detection-dataset/chest_xray" 
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")
OUTPUT_DIR = "/kaggle/working/outputs" 
os.makedirs(OUTPUT_DIR, exist_ok=True) 

# Path mapping and label encoding 
CLASS_MAP = {     
    "NORMAL": 0,     
    "PNEUMONIA_BACTERIAL": 1,     
    "PNEUMONIA_VIRAL": 2 
} 

class ChestXRayDataset(Dataset):      
    def __init__(self, root_dir, transform=None):         
        self.root_dir = root_dir        
        self.transform = transform        
        self.image_paths = []        
        self.labels = []                 
        
        for class_name, label in CLASS_MAP.items():            
            class_dir = os.path.join(root_dir, class_name)             
            if os.path.exists(class_dir):                 
                for fname in os.listdir(class_dir):                     
                    if fname.lower().endswith(('.png', '.jpg', '.jpeg')):                        
                        self.image_paths.append(os.path.join(class_dir, fname))                        
                        self.labels.append(label)                             
                        
    def __len__(self):          
        return len(self.image_paths)         
        
    def __getitem__(self, idx):          
        img_path = self.image_paths[idx]        
        image = Image.open(img_path).convert("RGB")        
        label = self.labels[idx]                 
        
        if self.transform:            
            image = self.transform(image)                     
            
        return image, label, img_path 

# Transformations featuring aggressive augmentation for training 
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# DataLoader construction 
train_dataset = ChestXRayDataset(TRAIN_DIR, transform=train_transforms)
test_dataset = ChestXRayDataset(TEST_DIR, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# ==============================================================================
# Phase II: Evidential Uncertainty Model (Swin-EViT)
# ==============================================================================

class SwinEViT(nn.Module):
    def __init__(self, num_classes=3):
        super(SwinEViT, self).__init__()
        # Initialize the Swin Transformer Base model from timm
        # Using num_classes=0 returns the raw 1024-D feature vector
        self.backbone = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=0)
        
        # Evidential Head replaces the standard linear classifier
        self.evidential_dense = nn.Linear(self.backbone.num_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        # Apply Softplus to ensure the generated evidence is strictly non-negative
        evidence = F.softplus(self.evidential_dense(features))
        return evidence

def kl_divergence_regularizer(alpha, num_classes):
    # Calculate KL Divergence between Dirichlet(alpha) and a uniform Dirichlet prior
    S = torch.sum(alpha, dim=1, keepdim=True)
    beta = torch.ones_like(alpha)
    S_beta = torch.sum(beta, dim=1, keepdim=True)
    
    lnB = torch.lgamma(S) - torch.sum(torch.lgamma(alpha), dim=1, keepdim=True)
    lnB_uni = torch.sum(torch.lgamma(beta), dim=1, keepdim=True) - torch.lgamma(S_beta)
    
    dg0 = torch.digamma(S)
    dg1 = torch.digamma(alpha)
    
    kl = torch.sum((alpha - beta) * (dg1 - dg0), dim=1, keepdim=True) + lnB + lnB_uni
    return kl

def evidential_loss(evidence, target, epoch, num_classes=3, annealing_step=10):
    # one-hot encode the target
    y = F.one_hot(target, num_classes=num_classes).float()
    
    # Calculate Dirichlet parameters
    alpha = evidence + 1
    S = torch.sum(alpha, dim=1, keepdim=True)
    
    # Expected Cross Entropy Loss
    ece_loss = torch.sum(y * (torch.digamma(S) - torch.digamma(alpha)), dim=1, keepdim=True)
    
    # KL Divergence Regularization
    # Strip away evidence from the correct class before computing KL
    alpha_tilde = y + (1 - y) * alpha
    kl_loss = kl_divergence_regularizer(alpha_tilde, num_classes)
    
    # Annealing coefficient to prevent early collapse into total uncertainty
    annealing_coef = min(1.0, epoch / annealing_step)
    
    total_loss = torch.mean(ece_loss + annealing_coef * kl_loss)
    return total_loss

# Initialize Model and Optimizer
model_evit = SwinEViT(num_classes=3).to(device)
optimizer_evit = optim.AdamW(model_evit.parameters(), lr=2e-5, weight_decay=1e-4)

# Training Loop for Phase II
def train_swin_evit(model, loader, optimizer, epochs=15):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for images, labels, _ in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # Utilize Automatic Mixed Precision for T4 GPU efficiency
            with torch.cuda.amp.autocast():
                evidence = model(images)
                loss = evidential_loss(evidence, labels, epoch)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
        print(f"Swin-EViT Epoch {epoch+1}/{epochs} | Evidential Loss: {epoch_loss/len(loader):.4f}")

# Execute training (Uncomment to run in Kaggle)
train_swin_evit(model_evit, train_loader, optimizer_evit, epochs=15)

In [ ]:
# ==============================================================================  
# Phase III: Generative XAI (CycleGAN Counterfactual Synthesizer)  
# ==============================================================================  
import itertools 
import torch
import torch.nn as nn
import torch.optim as optim

class ResidualBlock(nn.Module):      
    def __init__(self, channels):          
        super(ResidualBlock, self).__init__()        
        self.block = nn.Sequential(            
            nn.ReflectionPad2d(1),            
            nn.Conv2d(channels, channels, 3),            
            nn.InstanceNorm2d(channels),         
            nn.ReLU(inplace=True),            
            nn.ReflectionPad2d(1),            
            nn.Conv2d(channels, channels, 3),            
            nn.InstanceNorm2d(channels)        
        )     
        
    def forward(self, x):          
        return x + self.block(x) 

class GeneratorResNet(nn.Module):      
    def __init__(self, input_shape=(3, 224, 224), num_residual_blocks=9):          
        super(GeneratorResNet, self).__init__()        
        channels = input_shape[0]                 
        
        # Initial convolution block         
        out_features = 64         
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(channels, out_features, 7),
            nn.InstanceNorm2d(out_features),
            nn.ReLU(inplace=True)
        ]                 
        
        # Downsampling         
        in_features = out_features         
        for _ in range(2):            
            out_features *= 2             
            model += [
                nn.Conv2d(in_features, out_features, 3, stride=2, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]            
            in_features = out_features                     
            
        # Residual blocks maintaining spatial dimensions          
        for _ in range(num_residual_blocks):            
            model += [ResidualBlock(out_features)]                     
            
        # Upsampling          
        for _ in range(2):            
            out_features //= 2             
            model += [
                nn.ConvTranspose2d(in_features, out_features, 3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]            
            in_features = out_features              
            
        # Output layer         
        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, channels, 7),
            nn.Tanh()
        ]        
        
        self.model = nn.Sequential(*model)     
        
    def forward(self, x):          
        return self.model(x) 

class Discriminator(nn.Module):      
    def __init__(self, input_shape=(3, 224, 224)):          
        super(Discriminator, self).__init__()        
        channels, height, width = input_shape                
        
        def discriminator_block(in_filters, out_filters, normalize=True):             
            layers = [nn.Conv2d(in_filters, out_filters, 4, stride=2, padding=1)]             
            if normalize:                
                layers.append(nn.InstanceNorm2d(out_filters))            
            layers.append(nn.LeakyReLU(0.2, inplace=True))             
            return layers        
            
        self.model = nn.Sequential(            
            *discriminator_block(channels, 64, normalize=False),            
            *discriminator_block(64, 128),            
            *discriminator_block(128, 256),            
            *discriminator_block(256, 512),            
            nn.ZeroPad2d((1, 0, 1, 0)),            
            nn.Conv2d(512, 1, 4, padding=1)        
        )     
        
    def forward(self, img):          
        return self.model(img) 

# Assuming `device` is instantiated in Phase I. If running independently:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize CycleGAN Components 
G_XY = GeneratorResNet().to(device)  # Pneumonia to Normal 
G_YX = GeneratorResNet().to(device)  # Normal to Pneumonia 
D_X = Discriminator().to(device)     # Real vs Fake Pneumonia 
D_Y = Discriminator().to(device)     # Real vs Fake Normal  

# Loss functions 
criterion_GAN = nn.MSELoss().to(device)
criterion_cycle = nn.L1Loss().to(device)
criterion_identity = nn.L1Loss().to(device)

optimizer_G = optim.AdamW(itertools.chain(G_XY.parameters(), G_YX.parameters()), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_X = optim.AdamW(D_X.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D_Y = optim.AdamW(D_Y.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Note: In a full execution, the training loop involves fetching pairs of Normal (Y)   
# and Pneumonia (X) images, computing identity, GAN, and Cycle losses sequentially.

In [ ]:
# ==============================================================================  
# Phase IV: Evaluation Metrics and Difference Map Extraction  
# ==============================================================================  
from sklearn.metrics import classification_report, f1_score 
from torchmetrics import CalibrationError 
from torchmetrics.image.fid import FrechetInceptionDistance 
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

def evaluate_pipeline(model_evit, generator_XY, test_loader, uncertainty_threshold=0.2):     
    model_evit.eval()    
    generator_XY.eval()        
    
    # Initialize the empty lists required to store predictions and targets
    all_preds = []    
    all_targets = []    
    all_probs = []         
    
    # Initialize TorchMetrics for ECE and FID     
    ece_metric = CalibrationError(n_bins=15, norm='l1', task='multiclass', num_classes=3).to(device)    
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)         
    
    with torch.no_grad():         
        for images, labels, img_paths in test_loader:     
            images, labels = images.to(device), labels.to(device)                         
            
            # 1. Evidential Prediction             
            evidence = model_evit(images)            
            alpha = evidence + 1             
            S = torch.sum(alpha, dim=1, keepdim=True)                         
            
            # Compute probabilities and uncertainty             
            probs = alpha / S            
            uncertainty = 3.0 / S.squeeze()                        
            preds = torch.argmax(probs, dim=1)                         
            
            # 2. Selective Classification based on Uncertainty             
            valid_idx = uncertainty < uncertainty_threshold                         
            
            if valid_idx.any():                
                all_preds.extend(preds[valid_idx].cpu().numpy())                
                all_targets.extend(labels[valid_idx].cpu().numpy())                
                all_probs.append(probs[valid_idx])                                 
                
                # Update ECE calculation               
                ece_metric.update(probs[valid_idx], labels[valid_idx])                             
                
            # 3. Counterfactual Generation for identified Pneumonia cases (Label 1 or 2)
            pneumonia_idx = (preds > 0) & valid_idx
            
            if pneumonia_idx.any():
                infected_imgs = images[pneumonia_idx]
                
                # --- MODIFICATION 1: Grab corresponding labels for the infected images ---
                infected_labels = labels[pneumonia_idx] 
                
                # Update FID with real images (convert to uint8 0-255 format for FID metric)
                real_imgs_uint8 = ((infected_imgs + 1) / 2 * 255).byte()
                fid_metric.update(real_imgs_uint8, real=True)
                
                # Generate pseudo-healthy counterfactuals
                healthy_counterfactuals = generator_XY(infected_imgs)
                
                # Update FID with generated images
                fake_imgs_uint8 = ((healthy_counterfactuals + 1) / 2 * 255).byte()
                fid_metric.update(fake_imgs_uint8, real=False)
                
                # Generate and save Difference Maps
                for i in range(infected_imgs.size(0)):
                    orig_img = infected_imgs[i].cpu().numpy().transpose(1, 2, 0)
                    gen_img = healthy_counterfactuals[i].cpu().numpy().transpose(1, 2, 0)
                    
                    # De-normalize for visualization
                    orig_img = np.clip((orig_img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]), 0, 1)
                    gen_img = np.clip((gen_img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]), 0, 1)
                    
                    # Calculate absolute difference map
                    diff_map = np.abs(orig_img - gen_img)
                    diff_map_gray = np.mean(diff_map, axis=2)
                    
                    # --- MODIFICATION 2: Determine specific label string for the title ---
                    true_label = infected_labels[i].item()
                    infection_type = "Bacterial" if true_label == 1 else "Viral"
                    
                    # Save output plots to the Kaggle working directory
                    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                    
                    axes[0].imshow(orig_img)
                    # Use the dynamically generated f-string title
                    axes[0].set_title(f"Original Infected X-ray ({infection_type})")
                    axes[0].axis('off')
                    
                    axes[1].imshow(gen_img)
                    axes[1].set_title("Generated Pseudo-Healthy X-ray")
                    axes[1].axis('off')
                    
                    axes[2].imshow(diff_map_gray, cmap='jet')
                    axes[2].set_title("Counterfactual Difference Map")
                    axes[2].axis('off')
                    
                    save_path = os.path.join(OUTPUT_DIR, f"counterfactual_{len(all_preds)}_{i}.png")
                    plt.savefig(save_path, bbox_inches='tight')
                    plt.close(fig)  
                    
    # Compile Final Metrics     
    f1 = f1_score(all_targets, all_preds, average='weighted')    
    ece = ece_metric.compute().item()    
    fid = fid_metric.compute().item()        
    
    print(f"Weighted F1-Score (Confident Predictions): {f1:.4f}")    
    print(f"Expected Calibration Error (ECE): {ece:.4f}")    
    print(f"Fréchet Inception Distance (FID): {fid:.4f}") 

# Execute Evaluation (Uncomment to run in Kaggle)  
evaluate_pipeline(model_evit, G_XY, test_loader)